# Module 16 - Inference backends and production models

Use this notebook after `tests/test_inference.py` is passing. The notebook compares any saved course artifact you have with your configured ProdLM backend, benchmarks latency and throughput, adapts Module 15 generation eval to the new `Backend.complete(...)` interface, and includes optional extension cells for routing, KV-cache decoding, streaming, and MLX.

The deliverable is the inference-stack postmortem: which model/backend you will use for Modules 17-20, what it costs on your machine, and where the capability gap is between StudentLM/BaseLM artifacts and ProdLM.

## Setup

In [ ]:
from pathlib import Path
import gc
import json
import math
import subprocess
import sys
from typing import Any

import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display

from g2c.eval import GenerationExample, contains_match, run_generation_eval
from g2c.inference import (
    Backend,
    BackendInfo,
    InferenceResult,
    OllamaError,
    benchmark,
    load_artifact_backend,
    load_prodlm_backend,
    prodlm_manifest_exists,
)
from g2c.notebook_extras.eval import print_eval_report, print_generation_results
from g2c.notebook_extras.model_selection import select_inference_artifact_name
from g2c.notebook_extras.sampling import printable
from g2c.sampling import generate, generate_cached

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
print(repo_root)

Run the inference tests before proceeding. In the clean scaffold this cell should fail until you implement the Module 16 TODOs in `g2c/inference/`.

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_inference.py", "-q"],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Module 16 inference tests are not passing yet."

The optional KV-cache extension has its own tests. Skip the next cell if you are only doing the required backend work; it asserts on `pytest -k cached`, which fails until the cached-decoding paths are implemented.

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_multi_head_attention.py",
        "tests/test_transformer.py",
        "tests/test_sampling.py",
        "-k",
        "cached",
        "-q",
    ],
    cwd=repo_root,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, "Optional KV-cache tests are not passing yet."

## Model selection

BaseLM is the default artifact backend for this notebook. Artifact selection prefers the DPO stage when available, then SFT, then the base model. Set `ARTIFACT_SELECTION = "course"` to compare ProdLM against your strongest saved course artifact, or set it to an artifact base/name such as `"StoryLM-30M"` or `"StoryLM-30M-SFT"`.

In [ ]:
ARTIFACT_SELECTION = "BaseLM"  # "BaseLM", "course", or an artifact base/name such as "TinyLLM-30M"
LOAD_ARTIFACT = True
LOAD_PRODLM = True
ARTIFACT_DEVICE = "auto"
ARTIFACT_TORCH_DTYPE = "float16"

ARTIFACT_NAME = select_inference_artifact_name(
    ARTIFACT_SELECTION,
    repo_root=repo_root,
    load_artifact=LOAD_ARTIFACT,
)
print("selected artifact backend:", ARTIFACT_NAME)

## Load backends

The artifact backend uses the selected BaseLM/course artifact. ProdLM is loaded separately from the Ollama manifest if configured with `./prodlm.sh`.

In [ ]:
artifact_backend = None
if LOAD_ARTIFACT:
    try:
        artifact_backend = load_artifact_backend(
            ARTIFACT_NAME,
            repo_root=repo_root,
            device=ARTIFACT_DEVICE,
            torch_dtype=ARTIFACT_TORCH_DTYPE,
            required=False,
        )
    except Exception as exc:
        print(f"Artifact backend unavailable: {type(exc).__name__}: {exc}")

prodlm_backend = None
if LOAD_PRODLM:
    if prodlm_manifest_exists(repo_root=repo_root):
        prodlm_backend = load_prodlm_backend(repo_root=repo_root, required=True)
    else:
        print("ProdLM is not configured yet. Run ./prodlm.sh --model-id llama3.2:3b and rerun this cell.")

backends = [backend for backend in (artifact_backend, prodlm_backend) if backend is not None]

if not backends:
    print("No backends loaded yet. Run ./baselm.sh, save a model artifact, or configure ProdLM with ./prodlm.sh.")
else:
    for backend in backends:
        print(f"loaded {backend.info.name}: {backend.info.model_id}")
        if backend.info.extra:
            print("  extra:", backend.info.extra)

## Display helpers

In [ ]:
def fmt(value: Any, *, digits: int = 1) -> str:
    if value is None:
        return "-"
    if isinstance(value, float):
        return f"{value:.{digits}f}"
    return str(value)


def backend_label(backend: Backend) -> str:
    return f"{backend.info.name}/{backend.info.model_id}"


def complete_safely(
    backend: Backend,
    prompt: str,
    *,
    max_new_tokens: int = 80,
    temperature: float = 0.0,
    top_k: int | None = None,
    top_p: float | None = None,
) -> InferenceResult | None:
    try:
        return backend.complete(
            prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
        )
    except Exception as exc:
        print(f"[{backend_label(backend)}] failed: {type(exc).__name__}: {exc}")
        return None


def print_completion(result: InferenceResult | None) -> None:
    if result is None:
        return
    tps = result.tokens_per_second
    print(f"backend: {result.backend.name}/{result.backend.model_id}")
    print(f"latency: {result.latency_ms:.1f} ms | completion tokens: {fmt(result.completion_tokens, digits=0)} | tok/s: {fmt(tps)}")
    print("-" * 72)
    print(printable(result.completion).strip())


def markdown_table(rows: list[dict[str, Any]], columns: list[str]) -> None:
    if not rows:
        print("No rows.")
        return
    header = "| " + " | ".join(columns) + " |"
    sep = "| " + " | ".join("---" for _ in columns) + " |"
    body = []
    for row in rows:
        body.append("| " + " | ".join(str(row.get(col, "")) for col in columns) + " |")
    display(Markdown("\n".join([header, sep, *body])))


def benchmark_summary_row(name: str, result) -> dict[str, str]:
    return {
        "backend": name,
        "n": str(result.n),
        "mean ms": fmt(result.latency_ms_mean),
        "p50 ms": fmt(result.latency_ms_p50),
        "p90 ms": fmt(result.latency_ms_p90),
        "tokens": fmt(result.completion_tokens_total, digits=0),
        "tok/s": fmt(result.tokens_per_second_overall),
    }


def plot_benchmark_result(result, *, title: str | None = None) -> None:
    xs = list(range(1, result.n + 1))
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(xs, result.per_request_latency_ms, marker="o")
    axes[0].set_xlabel("request")
    axes[0].set_ylabel("latency ms")
    axes[0].set_title("latency")
    tps = [float("nan") if value is None else value for value in result.per_request_tokens_per_second]
    axes[1].plot(xs, tps, marker="o", color="#f58518")
    axes[1].set_xlabel("request")
    axes[1].set_ylabel("completion tok/s")
    axes[1].set_title("throughput")
    fig.suptitle(title or f"{result.backend.name}/{result.backend.model_id}")
    fig.tight_layout()
    plt.show()

## Exercise 1 - Smoke-test artifact and ProdLM backends

Run a few simple prompts through every loaded backend. This is not an eval yet; it is a first look at capability, latency, and whether each backend is wired correctly.

In [ ]:
SMOKE_PROMPTS = [
    "<|user|>\nWhat is the largest city in Spain?\n<|assistant|>\n",
    "<|user|>\nWhat is 13 + 28?\n<|assistant|>\n",
    "<|user|>\nIn one sentence, who wrote Hamlet?\n<|assistant|>\n",
    "<|user|>\nList three programming languages.\n<|assistant|>\n",
]

for backend in backends:
    print("=" * 88)
    print(backend_label(backend))
    for prompt in SMOKE_PROMPTS:
        print("\nPROMPT:", prompt.replace("\n", "\\n"))
        result = complete_safely(backend, prompt, max_new_tokens=80, temperature=0.0)
        print_completion(result)

## Exercise 2 - Benchmark ProdLM

This suite is intentionally mixed: short factual questions, arithmetic, formatting, and instruction-following. The first request may be a cold-start outlier. Look at p50 and p90, not just the mean.

In [ ]:
BENCHMARK_PROMPTS = [
    "What is the capital of France?",
    "What is the largest city in Spain?",
    "What is 13 + 28?",
    "What is 7 * 9?",
    "Write one sentence explaining photosynthesis.",
    "List three programming languages.",
    "Return JSON with keys name and color for a red apple.",
    "Summarize: Neural networks are functions with learned parameters.",
    "Answer yes or no: is water wet?",
    "Give one reason validation loss matters.",
    "What is the opposite of hot?",
    "Name the author of Hamlet.",
    "What gas do plants take in?",
    "What is 100 - 37?",
    "Rewrite in a friendly tone: Submit the report today.",
    "Give a two item checklist for debugging a failing test.",
    "What does CPU stand for?",
    "What is the next number: 2, 4, 6, 8?",
    "Explain overfitting in one sentence.",
    "Say only the word ready.",
]

benchmark_results = {}

if prodlm_backend is None:
    print("No ProdLM backend loaded. Configure with ./prodlm.sh and rerun the load cell.")
else:
    try:
        prodlm_benchmark = benchmark(
            prodlm_backend,
            BENCHMARK_PROMPTS,
            max_new_tokens=80,
            temperature=0.0,
            metadata={"suite": "module-16-prod-lm"},
        )
        benchmark_results["ProdLM"] = prodlm_benchmark
        print(prodlm_benchmark)
        markdown_table(
            [benchmark_summary_row("ProdLM", prodlm_benchmark)],
            ["backend", "n", "mean ms", "p50 ms", "p90 ms", "tokens", "tok/s"],
        )
        plot_benchmark_result(prodlm_benchmark, title="ProdLM benchmark")
    except Exception as exc:
        print(f"ProdLM benchmark failed: {type(exc).__name__}: {exc}")

## Exercise 3 - Optional MLX backend

This is an extension path, not a required deliverable. If you install `mlx-lm`, implement `MLXBackend` with the same `Backend.complete(...)` contract, then rerun the smoke and benchmark cells with it added to `backends`.

The next cell is a placeholder that raises `NotImplementedError`. Skip it unless you are taking on this extension, in which case replace the body with your `MLXBackend` implementation.

In [ ]:
raise NotImplementedError(
    "Implement MLXBackend as an extension: subclass Backend, wrap mlx_lm.generate, "
    "and return InferenceResult with prompt/completion/latency/tokens."
)

## Exercise 4 - Re-run a generation eval through ProdLM

Module 15's generation eval only needs a `generate_fn(prompt) -> str`. A backend adapter is therefore one small closure.

In [ ]:
generation_examples = [
    GenerationExample("Answer with only the city: What is the capital of France?", ["Paris"]),
    GenerationExample("Answer with only the number: What is 13 + 28?", ["41"]),
    GenerationExample("Answer with only the author name: Who wrote Hamlet?", ["Shakespeare", "William Shakespeare"]),
    GenerationExample("Answer yes or no: is the sky blue on a clear day?", ["yes"]),
    GenerationExample("Name one gas plants take in for photosynthesis.", ["carbon dioxide", "CO2"]),
]


def generate_fn_for_backend(
    backend: Backend,
    *,
    max_new_tokens: int = 40,
    temperature: float = 0.0,
):
    def generate_fn(prompt: str) -> str:
        result = backend.complete(
            prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
        )
        return result.completion.strip()

    return generate_fn


if prodlm_backend is None:
    print("No ProdLM backend loaded. Skipping generation eval.")
else:
    try:
        prodlm_eval = run_generation_eval(
            generation_examples,
            generate_fn_for_backend(prodlm_backend),
            contains_match,
            task_name="prodlm_generation_smoke",
        )
        print_eval_report(prodlm_eval, title="ProdLM generation eval")
        print_generation_results(generation_examples, prodlm_eval)
    except Exception as exc:
        print(f"ProdLM eval failed: {type(exc).__name__}: {exc}")

## Exercise 5 - Quantify the quantization tax

Use BaseLM to simulate what lower precision does to model behavior. This cell fake-quantizes weights to 8, 4, and 2 bits, then dequantizes them back into normal PyTorch tensors. That means it is a quality experiment, not a memory-saving runtime. Ollama/GGUF quantization is still the production path for actually reducing memory.

Requires the BaseLM artifact (run `./baselm.sh` first); the cell raises if BaseLM is missing. Each precision level reloads BaseLM, so expect a few minutes.

In [ ]:
BASELM_QUANT_BITS = [16, 8, 4, 2]
BASELM_QUANT_DTYPE = "float16"
BASELM_QUANT_MAX_NEW_TOKENS = 40


def fake_quantize_tensor_(tensor: torch.Tensor, bits: int) -> None:
    """Uniform symmetric fake quantization, dequantized back into tensor.dtype."""
    if bits >= 16 or not tensor.is_floating_point() or tensor.numel() == 0:
        return
    qmin = -(2 ** (bits - 1))
    qmax = (2 ** (bits - 1)) - 1
    if qmax <= 0:
        raise ValueError("bits must be at least 2")
    max_abs = tensor.detach().abs().max()
    if not torch.isfinite(max_abs) or max_abs.item() == 0:
        return
    scale = max_abs / max(abs(qmin), qmax)
    tensor.copy_((tensor / scale).round().clamp(qmin, qmax) * scale)


def fake_quantize_model_(model: torch.nn.Module, bits: int) -> None:
    with torch.no_grad():
        for parameter in model.parameters():
            fake_quantize_tensor_(parameter.data, bits)


def clear_torch_cache() -> None:
    gc.collect()
    if torch.backends.mps.is_available():
        torch.mps.empty_cache()


def load_baselm_for_quant() -> Backend:
    return load_artifact_backend(
        "BaseLM",
        repo_root=repo_root,
        device=ARTIFACT_DEVICE,
        torch_dtype=BASELM_QUANT_DTYPE,
        required=True,
    )


rows = []
for bits in BASELM_QUANT_BITS:
    backend = load_baselm_for_quant()
    if bits < 16:
        fake_quantize_model_(backend.model, bits)
    eval_report = run_generation_eval(
        generation_examples,
        generate_fn_for_backend(
            backend,
            max_new_tokens=BASELM_QUANT_MAX_NEW_TOKENS,
            temperature=0.0,
        ),
        contains_match,
        task_name=f"BaseLM fake {bits}-bit",
    )
    sample_prompt = "<|user|>\nAnswer yes or no: is the sky blue?\n<|assistant|>\n"
    sample = backend.complete(
        sample_prompt,
        max_new_tokens=BASELM_QUANT_MAX_NEW_TOKENS,
        temperature=0.0,
    ).completion
    rows.append(
        {
            "precision": BASELM_QUANT_DTYPE if bits == 16 else f"fake int{bits}",
            "accuracy": f"{eval_report.accuracy:.3f}",
            "sample": printable(sample).strip().replace("\n", " ")[:120],
        }
    )
    del backend
    clear_torch_cache()

markdown_table(rows, ["precision", "accuracy", "sample"])
print("Fake quantization changes quality only here. It does not reduce PyTorch model memory because weights stay as float tensors.")

## Exercise 6 - Build a router backend

This tiny router sends short prompts to the artifact backend and longer prompts to ProdLM. It is the same dispatching pattern that later agent systems use, except the routing condition here is deliberately simple.

In [ ]:
class RouterBackend(Backend):
    def __init__(self, short_backend: Backend, long_backend: Backend, *, threshold_tokens: int = 32) -> None:
        self.short_backend = short_backend
        self.long_backend = long_backend
        self.threshold_tokens = threshold_tokens
        self._info = BackendInfo(
            name="router",
            model_id=f"short={short_backend.info.model_id}|long={long_backend.info.model_id}",
            extra={"threshold_tokens": threshold_tokens},
        )

    @property
    def info(self) -> BackendInfo:
        return self._info

    def _prompt_tokens(self, prompt: str) -> int:
        tokenizer = getattr(self.short_backend, "tokenizer", None)
        if tokenizer is not None:
            try:
                return len(tokenizer.encode(prompt))
            except Exception:
                pass
        return len(prompt.split())

    def complete(self, prompt: str, *, max_new_tokens: int = 128, temperature: float = 1.0, top_k: int | None = None, top_p: float | None = None) -> InferenceResult:
        n_tokens = self._prompt_tokens(prompt)
        chosen = self.short_backend if n_tokens < self.threshold_tokens else self.long_backend
        result = chosen.complete(
            prompt,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
        )
        result.metadata["router"] = {
            "prompt_tokens_estimate": n_tokens,
            "threshold_tokens": self.threshold_tokens,
            "routed_to": backend_label(chosen),
        }
        result.backend = self.info
        return result


if artifact_backend is None or prodlm_backend is None:
    print("Router demo needs both an artifact backend and a ProdLM backend.")
else:
    router_backend = RouterBackend(artifact_backend, prodlm_backend, threshold_tokens=32)
    for prompt in SMOKE_PROMPTS:
        result = complete_safely(router_backend, prompt, max_new_tokens=64, temperature=0.0)
        print_completion(result)
        if result is not None:
            print("route:", result.metadata.get("router"))

## Exercise 7 - Optional toy KV cache

The production path uses Ollama's cache. This optional demo checks that your course model's `generate_cached` path matches regular greedy generation for the same prompt.

In [ ]:
RUN_KV_CACHE_DEMO = artifact_backend is not None and hasattr(artifact_backend.model, "forward_cached")

if not RUN_KV_CACHE_DEMO:
    print("KV-cache demo needs a course TransformerLM artifact with forward_cached.")
else:
    prompt = "Once upon a time,"
    tokenizer = artifact_backend.tokenizer
    prompt_ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long)
    eos_id = getattr(artifact_backend, "_eos_id", None)
    uncached = generate(
        artifact_backend.model,
        prompt_ids,
        max_new_tokens=40,
        temperature=0.0,
        eos_id=eos_id,
    )
    cached = generate_cached(
        artifact_backend.model,
        prompt_ids,
        max_new_tokens=40,
        temperature=0.0,
        eos_id=eos_id,
    )
    print("same token ids:", torch.equal(uncached, cached))
    print("uncached length:", len(uncached), "cached length:", len(cached))
    print("sample:")
    print(printable(tokenizer.decode([int(x) for x in cached.tolist()])))

## Exercise 8 - Optional streaming

Streaming changes the response protocol: Ollama returns newline-delimited JSON chunks instead of one JSON object. Implement this as a separate method such as `complete_stream` rather than complicating the required `complete` contract.

The next cell is a placeholder that raises `NotImplementedError`. Skip it unless you are taking on this extension, in which case replace the body with your streaming implementation.

In [ ]:
raise NotImplementedError(
    "Subclass OllamaBackend, send stream=True, iterate response lines, "
    "json.loads each line, and yield each chunk's response text."
)

## Exercise 9 - Inference-stack postmortem

Use the template below as a starting point. The deliverable should be evidence-backed: model IDs, latency/throughput numbers, capability gaps, and the default backend you plan to use for Modules 17-20.

In [ ]:
SAVE_POSTMORTEM_DRAFT = False
POSTMORTEM_PATH = repo_root / "docs" / "inference-postmortem.md"

postmortem_draft = """# Module 16 inference-stack postmortem

## What I ran

- Artifact backend:
- ProdLM backend:
- Machine:
- Quantization / model tags:
- Benchmark suite size:

## Throughput and latency

| backend | mean ms | p50 ms | p90 ms | tok/s |
| --- | ---: | ---: | ---: | ---: |
| artifact | | | | |
| ProdLM | | | | |

## Capability gaps

Describe which prompts the artifact handled, which prompts only ProdLM handled, and which prompts remained weak.

## Default backend for Modules 17-20

I will use ... because ...
"""

print(postmortem_draft)

if SAVE_POSTMORTEM_DRAFT:
    POSTMORTEM_PATH.write_text(postmortem_draft, encoding="utf-8")
    print("wrote", POSTMORTEM_PATH)